# Digit Prediction

We try to identify Digits from 28x28 pixel image in greyscale.

In [1]:
%pip install scikit-learn
%pip install pandas 
%pip install seaborn
%pip install tensorflow
%pip install xgboost
%pip install lightgbm
%pip install pygad
%pip install scikeras



Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Imports
import pandas as pd


import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
from sklearn import preprocessing, svm 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression 
from sklearn import datasets, linear_model
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score,root_mean_squared_log_error


In [3]:
train_url = './digits/train.csv'
test_url = './digits/test.csv'
train = pd.read_csv(train_url)
test = pd.read_csv(test_url)

print("Training dataset shape", train.shape)
print("Test dataset shape", test.shape)


Training dataset shape (42000, 785)
Test dataset shape (28000, 784)


In [4]:
# Split train dataset 
feature_cols = train.columns[1:]
print("Features num=", len(feature_cols), feature_cols)
small_train = train.sample(frac=0.8,random_state=200)

X=small_train[feature_cols]
Y=small_train.loc[:, ['label']]
Xval=train.drop(X.index)[feature_cols]
Yval=train.drop(X.index).loc[:, ['label']]

print("Training dataset shape", X.shape, Y.shape)
print("Validation dataset shape", Xval.shape, Yval.shape)


Features num= 784 Index(['pixel0', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6',
       'pixel7', 'pixel8', 'pixel9',
       ...
       'pixel774', 'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779',
       'pixel780', 'pixel781', 'pixel782', 'pixel783'],
      dtype='object', length=784)


Training dataset shape (33600, 784) (33600, 1)
Validation dataset shape (8400, 784) (8400, 1)


In [5]:
def build_decision_tree(max_features):
    print("Decision tree...")
    # Writing a small decision tree classifier
    from sklearn import tree
    clf = tree.DecisionTreeClassifier(max_features=max_features)
    clf = clf.fit(X, Y)
    #tree.plot_tree(clf)
    return clf

In [6]:
def build_svc():
    print("SVC...")
    from sklearn import svm
    from sklearn.svm import LinearSVC
    clf = svm.SVC(kernel='rbf', probability=True)
    clf.fit(X, Y)
    return clf

In [7]:
def build_nearest_centroid():
    print("NearestCentroid...")
    from sklearn.neighbors import NearestCentroid
    clf = NearestCentroid()
    clf.fit(X, Y)
    return clf

In [8]:

from sklearn.neighbors import KNeighborsClassifier
def build_knn(n_neighbors: int = 2):
    print("KNeighborsClassifier...")

    clf =  KNeighborsClassifier(n_neighbors=n_neighbors)  
    clf.fit(X, Y)
    return clf

In [9]:

from sklearn.neighbors import KNeighborsClassifier, NeighborhoodComponentsAnalysis
from sklearn.pipeline import Pipeline
def build_complex_knn(n_neighbors: int = 3):
    print("ComplexKNeighborsClassifier...")
    nca = NeighborhoodComponentsAnalysis(random_state=42)

    knn = KNeighborsClassifier(n_neighbors=n_neighbors)

    nca_pipe = Pipeline([('nca', nca), ('knn', knn)])

    nca_pipe.fit(X, Y)
    return nca_pipe

In [10]:
from sklearn.ensemble import AdaBoostClassifier
def build_ada(estimators=200):
    clf = AdaBoostClassifier(n_estimators=estimators, algorithm="SAMME")
    clf.fit(X,Y)
    return clf

In [11]:
def build_bagging_classifier():
    from sklearn.ensemble import BaggingClassifier
    model = BaggingClassifier(n_estimators=20)
    model.fit(X, Y)
    return model
    

In [12]:
from sklearn.ensemble import HistGradientBoostingClassifier


def build_HistGradientBoostingClassifier(iter=20):
    print("build_HistGradientBoostingClassifier...")
    clf = HistGradientBoostingClassifier(max_iter=iter).fit(X, Y)
    return clf

In [13]:
# Create a function that evaluates a classifier and produce the metrics
from sklearn.metrics import accuracy_score,f1_score,recall_score, precision_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix

def evaluate_model(model, modelName: str):
    print("Prediction for ", modelName)
    prediction = model.predict(Xval)
    acc = accuracy_score( Yval, prediction)
    print(modelName, "Accuracy", acc)
    
    #matrix = confusion_matrix(Yval, prediction)
    #print(matrix)
    #disp = ConfusionMatrixDisplay(confusion_matrix=matrix) 
    # Then just plot it: 
    #disp.plot() 
    # And show it: 
    #plt.show()    
    
    print(modelName, "F1", f1_score(Yval, prediction, average="macro"))
    print(modelName, "Recall", recall_score(Yval, prediction, average="macro"))
    print(modelName, "Precision", precision_score(Yval, prediction, average="macro"))
    

In [14]:
# Test Decision tree accuracy
k_model = build_knn()
evaluate_model(k_model, "knn")


KNeighborsClassifier...


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Prediction for  knn
knn Accuracy 0.9610714285714286
knn F1 0.960585485798733
knn Recall 0.9602799685182963
knn Precision 0.9619758704314505


In [15]:

nc_model = build_nearest_centroid()
evaluate_model(nc_model, "nearest_centroid")


NearestCentroid...


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Prediction for  nearest_centroid
nearest_centroid Accuracy 0.8086904761904762
nearest_centroid F1 0.8069959351056537
nearest_centroid Recall 0.8052873849247553
nearest_centroid Precision 0.8125884675387727


In [16]:

#svc_model = build_svc()
hgb_model = build_HistGradientBoostingClassifier()
evaluate_model(hgb_model, "hgb_model")


build_HistGradientBoostingClassifier...


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Prediction for  hgb_model
hgb_model Accuracy 0.9495238095238095
hgb_model F1 0.9491055144945406
hgb_model Recall 0.949122583037021
hgb_model Precision 0.949154342760021


In [17]:

ada_model = build_ada()

evaluate_model(ada_model, "ada")


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Prediction for  ada
ada Accuracy 0.7533333333333333
ada F1 0.7530364370950469
ada Recall 0.7504401216100505
ada Precision 0.7629070599528344


In [18]:

bag_model = build_bagging_classifier()
evaluate_model(bag_model, "bag_model")


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/ensemble/_bagging.py:888: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Prediction for  bag_model
bag_model Accuracy 0.9454761904761905
bag_model F1 0.9448953926786091
bag_model Recall 0.9450154673991171
bag_model Precision 0.9448902144020996


In [19]:
#max_features = [ 10, 40, 100, 150, 200, 400, 800 ]
#for n in max_features:
d_model = build_decision_tree(None)
evaluate_model(d_model, "decision_tree" )


Decision tree...


Prediction for  decision_tree
decision_tree Accuracy 0.86
decision_tree F1 0.8581834604853864
decision_tree Recall 0.858056351215799
decision_tree Precision 0.8585214628316115


: 

In [20]:

cknn_model = build_complex_knn()
evaluate_model(cknn_model, "complex_knn")


ComplexKNeighborsClassifier...


/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:

from sklearn.ensemble import VotingClassifier
eclf = VotingClassifier(estimators=[('dt', d_model), ('knn', k_model), ('nc', nc_model), ('hgb', hgb_model), ('ada', ada_model)])
eclf.fit(X,Y)
evaluate_model(eclf, "eclf")

In [ ]:

from sklearn.ensemble import VotingClassifier
eclf = VotingClassifier(estimators=[('dt', d_model), ('knn', k_model), ('nc', nc_model), ('ada', ada_model)])
eclf.fit(X,Y)   
evaluate_model(eclf, "eclf")

/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


Prediction for  eclf
eclf Accuracy 0.9151190476190476
eclf F1 0.9139846813986626
eclf Recall 0.9134294793963995
eclf Precision 0.9160538294224375


In [ ]:
# StackingClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier


def stack_classifier():
    estimators = [('dt', d_model), ('knn', k_model), ('nc', nc_model), ('ada', ada_model)]
    
    final_layer_rfr = RandomForestClassifier(
        n_estimators=10, max_features=1, max_leaf_nodes=5,random_state=42)
    final_layer_gbr = GradientBoostingClassifier(
        n_estimators=10, max_features=1, max_leaf_nodes=5,random_state=42)
    final_layer = StackingClassifier(
        estimators=[('rf', final_layer_rfr),
                ('gbrt', final_layer_gbr)])
    
    clf = StackingClassifier(estimators=estimators, final_estimator=final_layer)
    clf.fit(X, Y)
    return clf

stack_clf = stack_classifier()

/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


KeyboardInterrupt: 

In [ ]:
import numpy as np
from scikeras.wrappers import KerasClassifier, KerasRegressor
import keras
from sklearn.model_selection import GridSearchCV
import tensorflow.keras as kr 
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt



def build_tensorflow_model(init='normal', display=False):
    
    Xgb = X / 255.0
    Xvalgb = Xval / 255.0

    print(Xgb.shape)

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(784, activation='relu'),
        tf.keras.layers.Dense(784, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam',
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=['accuracy'])
    if display:
        model.summary()

    return model

# Create a KerasClassifier wrapper for Scikit-learn
model = KerasClassifier(build_fn=build_tensorflow_model, epochs=10, batch_size=32, verbose=0)

# Define the hyperparameters to tune
param_grid = {
    'optimizer': ['adam', 'sgd', 'rmsprop'],
    'activation': ['relu', 'tanh'],
    'batch_size': [16, 32, 64, 100, 1000],
    'epochs': [3, 5, 10, 20]
}

# Use GridSearchCV to find the best combination of hyperparameters
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)
grid_result = grid.fit(X, Y)

# Display the best hyperparameters and corresponding accuracy
print(f"Best Parameters: {grid_result.best_params_}")
print(f"Best Accuracy: {grid_result.best_score_}")
tensor_model = grid



ImportError: cannot import name 'KerasClassifier' from 'scikeras.wrappers' (/home/sleroy/git/ml/venv/lib/python3.12/site-packages/scikeras/wrappers.py)

In [ ]:
def generate_prediction_test_tensor(model):
  dnn_predict = model.predict(test)  
  top_class_indices = np.argmax(dnn_predict, axis=1)
  df = pd.DataFrame({'Best_Class_ID':   top_class_indices})
  print(df)
  submission_df = pd.DataFrame()
  submission_df['Label'] = top_class_indices
  submission_df['ImageId'] = range(1, len(submission_df)+1 )
  submission_df.to_csv('digit_submission.csv', index=False) 

generate_prediction_test_tensor(tensor_model) 

875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step
       Best_Class_ID
0                  2
1                  0
2                  9
3                  9
4                  3
...              ...
27995              9
27996              7
27997              3
27998              9
27999              2

[28000 rows x 1 columns]


In [ ]:
# StackingClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier


def stack_classifier():
    estimators = [('dt', d_model), ('knn', k_model), ('nc', nc_model), ('ada', ada_model)]
    
    final_layer_rfr = RandomForestClassifier(
        n_estimators=10, max_features=1, max_leaf_nodes=5,random_state=42)
    final_layer_gbr = GradientBoostingClassifier(
        n_estimators=10, max_features=1, max_leaf_nodes=5,random_state=42)
    final_layer = StackingClassifier(
        estimators=[('rf', final_layer_rfr),
                ('gbrt', final_layer_gbr)])
    
    clf = StackingClassifier(estimators=estimators, final_estimator=final_layer)
    clf.fit(X, Y)
    return clf

stack_clf = stack_classifier()

/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


KeyboardInterrupt: 

In [ ]:
# StackingClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier


def stack_classifier():
    estimators = [('dt', d_model), ('knn', k_model), ('nc', nc_model), ('ada', ada_model)]
    
    final_layer_rfr = RandomForestClassifier(
        n_estimators=10, max_features=1, max_leaf_nodes=5,random_state=42)
    final_layer_gbr = GradientBoostingClassifier(
        n_estimators=10, max_features=1, max_leaf_nodes=5,random_state=42)
    final_layer = StackingClassifier(
        estimators=[('rf', final_layer_rfr),
                ('gbrt', final_layer_gbr)])
    
    clf = StackingClassifier(estimators=estimators, final_estimator=final_layer)
    clf.fit(X, Y)
    return clf

stack_clf = stack_classifier()

/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/sleroy/git/ml/venv/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


KeyboardInterrupt: 

In [ ]:
def generate_prediction_test(model):
  kaggle_predict = model.predict(test)  
  submission_df = pd.DataFrame()
  submission_df['Label'] = kaggle_predict
  submission_df['ImageId'] = range(1, len(submission_df)+1 )
  submission_df.to_csv('digit_submission.csv', index=False) 

generate_prediction_test(d_model) 

<Sequential name=sequential_3, built=True>
875/875 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step


ValueError: Cannot set a frame with no defined index and a value that cannot be converted to a Series